# <span style="color:darkblue"> Lecture 18: Machine Learning Pipelines </span>

<font size = "5">


One of the most common mistakes in ML analyses is data leakage:

https://www.youtube.com/watch?v=y8qaI5mpJeA

This happens when you use information from leak information from the <br>
validation/test sets to the training set.
- This can happen when we proprocess X using the whole dataset
- Machine learning pipelines automate the pre-processing stages and <br>
help avoid data leakage


In [1]:
603 365 4626

SyntaxError: invalid syntax (3127049958.py, line 1)

# <span style="color:darkblue"> I. Setup Working Environment </span>


In [2]:
# Import the package for the University of California Irvine API
from ucimlrepo import fetch_ucirepo 

# Import SK-Learn library for machine learning functions
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import get_scorer_names

# Import ML pipeline:
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

# Import PCA
from sklearn.decomposition import PCA

# Import standard data analysis packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Kmenas
from sklearn.cluster import MiniBatchKMeans



<font size = "5">

Cancer screening dataset


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

# <span style="color:darkblue"> II. Pipelines </span>

<font size = "5">

Example of machine learning pipeline

- Involves multiple splitting + pre-processing steps
- Preprocessing steps involve ```.fit()``` and ```.transform()```
- Model fitting at the end only involves ```.fit()```
- Evaluate with a scorer out-of-sample

In [4]:
# Split into training/test
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    random_state=0)

# Scale inputs
scaler        = StandardScaler().fit(X_train)
X_train_star  = scaler.transform(X_train)

# Obtain PCA (optional)
pca           = PCA(n_components=3).fit(X_train_star)
X_train_star  = pca.transform(X_train_star)

# Fit SVC model 
svm = SVC()
svm.fit(X_train_star, y_train)

# Predict test data
# Note: Here we are scaling and obtaining based on the loadings
# from the training data (we could also do this for the test data)
X_test_star = pca.transform(scaler.transform(X_test))

# Evaluate prediction accuracy
svm.score(X_test_star,y_test)

# Optional: Obtain predictions
# predictions = svm.predict(X_test_star)

0.916083916083916

<font size = "5">

Automatic ML pipelines

- Chain multiple functions together, applied in order
- Specify names of functions and outputs.
- The last step uses ```.fit()```, while the intermediate <br>
steps use ```.fit()``` and ```.transform()```. Any set of functions <br>
with this structure can be used with ```Pipeline()```

In [5]:
# Main syntax
# Each step is a different element of a list
# Each step is comprised of two arguments: name + function
# (wrapped in parentheses and comma-separated)
pipe = Pipeline([("scaler",StandardScaler()),
                 ("pca",PCA(n_components= 3)),
                 ("svm",SVC())])

  
pipe.fit(X_train,y_train)
pipe.score(X_test,y_test)


0.916083916083916

<font size = "5">

Abbreviated pipeline syntax

- Useful if you don't need to reference one of the components <br>
in subsequent analyses


In [6]:
# We can also use the abbreviate syntax
pipe_abbreviated = make_pipeline(StandardScaler(),
                                 PCA(n_components = 3),
                                 SVC())

<font size = "5">

Try it yourself!

Check that the abbreviated syntax also fits a model <br>
with the same prediction accuracy

In [7]:

pipe_abbreviated.fit(X_train,y_train)
pipe_abbreviated.score(X_test,y_test)


0.916083916083916

<font size = "5">

Try it yourself

Create a pipeline without PCA. Does it have better accuracy?

In [8]:
# Write your own code

pipe_no_pca = Pipeline([("scaler",StandardScaler()),
                 # ("pca",PCA(n_components= 3)),
                 ("svm",SVC())])

  
pipe_no_pca.fit(X_train,y_train)
pipe_no_pca.score(X_test,y_test)



0.965034965034965

# <span style="color:darkblue"> II. Pipelines + Cross-Validation </span>

<font size = "5">

Create a grid of values to search

- In the pipe framework, use name of the step, e.g. "svm" <br>
followed by two underscores, and the name of the parameters
- Doesn't work for abbreviated pipes

In [12]:
k_features = 3
# svm + __ + the parameter
param_grid = {'svm__C':     [ 0.01, 0.1],
              'svm__gamma': [0.01/(k_features), 0.1/(k_features)]}


<font size = "5">

Create a CV grid search

- The advantage is that the scaling and PCA steps are done on <br>
the spit-sample folds.
- Pipes avoid a proble known as "leakage": <br>
incorrectly using scale/PCA using the whole data <br>
violates the assumption of independence of the folds <br>
and could we obtain distorted evaluation metrics



In [11]:
# Configure grid
grid = GridSearchCV(pipe, param_grid, cv=2,
                          return_train_score=True,
                          n_jobs = -1)

grid.fit(X_train,y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...svm', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svm__C': [0.01, 0.1], 'svm__gamma': [0.0033333333333333335, 0.03333333333333333]}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined 

<font size = "5">

Evaluate cross-validated model

In [93]:
print("Best estimator:\n{}".format(grid.best_estimator_))


grid.score(X_test,y_test)

Best estimator:
Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=3)),
                ('svm', SVC(C=0.1, gamma=0.03333333333333333))])


0.9300699300699301

<font size = "5">

Try it yourself

Cross-validate and estimate a decision tree, where you try out different pruning penalties.

In [1]:
# Write your own code






# <span style="color:darkblue"> II. Pipelines + Cross-Validation + Multimodels </span>

<font size = "5">

Define names of each step, and a default option:

In [94]:
pipe_multimodel = Pipeline([('preprocessing', StandardScaler()),
                            ('classifier', SVC())])


<font size = "5">

Define grid of models/tuning parameters

- Can vary which classifier
- Can vary whether to use preprocessing or not
- Within a specific model


In [95]:
# Each separate model is grouped with curly brackets

param_grid = [
    {'classifier': [SVC()], 'preprocessing': [StandardScaler(), None],
     'classifier__C':   [ 0.01, 0.1]},
    {'classifier': [RidgeClassifier()], 'preprocessing': [StandardScaler(), None]}]

<font size = "5">

Search over all models

In [96]:
# Configure grid
grid = GridSearchCV(pipe_multimodel, param_grid, cv=2,
                          return_train_score=True,
                          n_jobs = -1)

grid.fit(X_train,y_train)

print("Best estimator:\n{}".format(grid.best_estimator_))

print(grid.score(X_test,y_test))

Best estimator:
Pipeline(steps=[('preprocessing', StandardScaler()),
                ('classifier', RidgeClassifier())])
0.965034965034965


<font size = "5">

Try it yourself

Cross-validate and estimate a decision tree, where you try out different pruning penalties.

In [ ]:
# Write your own code




